# SpendLens v3 — Procurement Intelligence Dashboard

**What's new in v3:**
- Storytelling layout: 5 sections each answering one question
- 5 clickable KPI cards in header (Total Spend, YoY Growth, EBITDA Impact, Contract Coverage, Maverick %)
- Vertical battery diagrams for procurement health KPIs
- Expiring contracts table (top 10, contract value vs actual spend)
- EBITDA Impact section (savings + cost avoidance + budget variance)
- Revised data model with budget, PO coverage, savings, contract value fields
- Deep dive tab for analysts (CAGR, treemap, capex/opex, gauges)


## 0. Setup
Run once. Skip if packages already installed (venv already has them).

In [ ]:
!pip install panel holoviews hvplot plotly pandas openpyxl bokeh param anthropic --quiet
print("✅ Done")

## 1. Imports

In [ ]:
import panel as pn
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import io, os, re, json
from datetime import datetime, date

pn.extension("plotly", sizing_mode="stretch_width")
print(f"✅ Panel {pn.__version__} | Pandas {pd.__version__}")

## 2. Theme — Navy / White Enterprise

In [ ]:
# ── Palette ──
BG      = "#FFFFFF"
CARD    = "#F8F9FA"
GRID    = "#E9ECEF"
TEXT    = "#1A1A2E"
DIM     = "#6C757D"
NAVY    = "#1B3A6B"
NAVY2   = "#2E5BA8"
GREEN   = "#1A7A4A"
YELLOW  = "#B8860B"
RED     = "#C0392B"
ORANGE  = "#D35400"

COLORS = ["#1B3A6B","#2E5BA8","#1A7A4A","#B8860B","#6A3EA1",
          "#0E7C86","#C0392B","#D35400","#2C7873","#4A235A"]

RISK_COLORS = {"Critical": RED, "High": YELLOW, "Medium": ORANGE, "Low": GREEN}

# Traffic light for KPIs
def traffic_color(value, target, lower_is_better=False):
    """Returns color based on how close value is to target."""
    ratio = value / target if target > 0 else 0
    if lower_is_better:
        if value <= target: return GREEN
        elif value <= target * 1.5: return YELLOW
        else: return RED
    else:
        if ratio >= 0.95: return GREEN
        elif ratio >= 0.75: return YELLOW
        else: return RED

CSS = """
body { background-color: #FFFFFF !important; color: #1A1A2E !important; }
.bk-root { background-color: #FFFFFF !important; }
"""
pn.config.raw_css.append(CSS)
print("🎨 Theme ready")

## 3. Data Model
Full SaaS taxonomy with budget, PO coverage, savings, contract value fields.

In [ ]:
YEARS     = [2022, 2023, 2024, 2025, 2026]
HEADCOUNT = [49, 120, 220, 380, 600]

categories_raw = [
    {"name": "Cloud & Compute",
     "spend": [4200,7800,12500,17800,24000],
     "budget_2026e": 22000, "concentration": 72, "risk": "Critical",
     "single_source": False, "lead_time_days": 0, "contract_end": "2026-09",
     "capex_opex": "Opex", "region": "US", "supplier_count": 3,
     "po_coverage_pct": 95, "contract_coverage_pct": 92,
     "savings_achieved": 800, "cost_avoidance": 400,
     "contract_value": 26000,
     "suppliers": [
         {"name": "Amazon Web Services", "pct": 60, "contract_end": "2026-09", "contract_value": 15000},
         {"name": "Google Cloud Platform","pct": 25, "contract_end": "2027-03", "contract_value": 6500},
         {"name": "Microsoft Azure",      "pct": 15, "contract_end": "2026-12", "contract_value": 4500},
     ]},
    {"name": "AI/ML APIs & Data",
     "spend": [800,2200,4800,6500,9200],
     "budget_2026e": 8500, "concentration": 55, "risk": "High",
     "single_source": False, "lead_time_days": 14, "contract_end": "2026-06",
     "capex_opex": "Opex", "region": "US", "supplier_count": 5,
     "po_coverage_pct": 70, "contract_coverage_pct": 75,
     "savings_achieved": 0, "cost_avoidance": 320,
     "contract_value": 9500,
     "suppliers": [
         {"name": "OpenAI",       "pct": 40, "contract_end": "2026-06", "contract_value": 3800},
         {"name": "Anthropic",    "pct": 25, "contract_end": "2026-12", "contract_value": 2300},
         {"name": "Scale AI",     "pct": 20, "contract_end": "2027-01", "contract_value": 1900},
         {"name": "Hugging Face", "pct": 10, "contract_end": "2026-09", "contract_value": 920},
         {"name": "Cohere",       "pct":  5, "contract_end": "2026-08", "contract_value": 460},
     ]},
    {"name": "IT Software & SaaS",
     "spend": [900,1400,2200,3100,4200],
     "budget_2026e": 4000, "concentration": 22, "risk": "Low",
     "single_source": False, "lead_time_days": 7, "contract_end": "Various",
     "capex_opex": "Opex", "region": "Global", "supplier_count": 18,
     "po_coverage_pct": 55, "contract_coverage_pct": 70,
     "savings_achieved": 280, "cost_avoidance": 0,
     "contract_value": 4400,
     "suppliers": [
         {"name": "GitHub / Copilot",   "pct": 22, "contract_end": "2026-11", "contract_value": 924},
         {"name": "Datadog",            "pct": 18, "contract_end": "2027-01", "contract_value": 756},
         {"name": "Atlassian",          "pct": 15, "contract_end": "2026-10", "contract_value": 630},
         {"name": "Slack / Salesforce", "pct": 12, "contract_end": "2026-12", "contract_value": 504},
         {"name": "Notion",             "pct":  8, "contract_end": "2026-09", "contract_value": 336},
         {"name": "Other SaaS tools",   "pct": 25, "contract_end": "Various", "contract_value": 1050},
     ]},
    {"name": "Telecom & Voice Infra",
     "spend": [400,800,1400,2200,3000],
     "budget_2026e": 2800, "concentration": 65, "risk": "Critical",
     "single_source": True, "lead_time_days": 30, "contract_end": "2026-07",
     "capex_opex": "Opex", "region": "US", "supplier_count": 3,
     "po_coverage_pct": 80, "contract_coverage_pct": 88,
     "savings_achieved": 220, "cost_avoidance": 0,
     "contract_value": 3200,
     "suppliers": [
         {"name": "Twilio",            "pct": 65, "contract_end": "2026-07", "contract_value": 2080},
         {"name": "Vonage / Ericsson", "pct": 25, "contract_end": "2027-02", "contract_value": 800},
         {"name": "Deutsche Telekom",  "pct": 10, "contract_end": "2027-12", "contract_value": 320},
     ]},
    {"name": "Recruitment & HR",
     "spend": [600,1100,2400,4200,6800],
     "budget_2026e": 6500, "concentration": 35, "risk": "High",
     "single_source": False, "lead_time_days": 45, "contract_end": "2026-12",
     "capex_opex": "Opex", "region": "DACH", "supplier_count": 8,
     "po_coverage_pct": 45, "contract_coverage_pct": 65,
     "savings_achieved": 180, "cost_avoidance": 0,
     "contract_value": 7200,
     "suppliers": [
         {"name": "Personio (HRIS)",    "pct": 20, "contract_end": "2027-03", "contract_value": 1360},
         {"name": "LinkedIn Talent",    "pct": 35, "contract_end": "2026-12", "contract_value": 2380},
         {"name": "Kienbaum / Kearney", "pct": 15, "contract_end": "2026-09", "contract_value": 1020},
         {"name": "Staffing agencies",  "pct": 20, "contract_end": "Various",  "contract_value": 1360},
         {"name": "Other job boards",   "pct": 10, "contract_end": "Various",  "contract_value": 680},
     ]},
    {"name": "Professional Services",
     "spend": [400,700,1200,2100,3200],
     "budget_2026e": 3000, "concentration": 40, "risk": "Medium",
     "single_source": False, "lead_time_days": 21, "contract_end": "2026-06",
     "capex_opex": "Opex", "region": "DACH", "supplier_count": 7,
     "po_coverage_pct": 60, "contract_coverage_pct": 80,
     "savings_achieved": 150, "cost_avoidance": 0,
     "contract_value": 3500,
     "suppliers": [
         {"name": "KPMG",                   "pct": 30, "contract_end": "2026-06", "contract_value": 1050},
         {"name": "Deloitte",               "pct": 25, "contract_end": "2026-03", "contract_value": 875},
         {"name": "EY",                     "pct": 15, "contract_end": "2026-09", "contract_value": 525},
         {"name": "Baker McKenzie (Legal)",  "pct": 20, "contract_end": "2026-12", "contract_value": 700},
         {"name": "Implementation partners","pct": 10, "contract_end": "Various",  "contract_value": 350},
     ]},
    {"name": "Marketing & Campaigns",
     "spend": [300,800,1800,3500,5500],
     "budget_2026e": 5000, "concentration": 28, "risk": "Medium",
     "single_source": False, "lead_time_days": 30, "contract_end": "Various",
     "capex_opex": "Opex", "region": "DACH", "supplier_count": 10,
     "po_coverage_pct": 40, "contract_coverage_pct": 55,
     "savings_achieved": 0, "cost_avoidance": 0,
     "contract_value": 5500,
     "suppliers": [
         {"name": "Google Ads",       "pct": 28, "contract_end": "Rolling", "contract_value": 1540},
         {"name": "LinkedIn Ads",     "pct": 22, "contract_end": "Rolling", "contract_value": 1210},
         {"name": "Event agencies",   "pct": 20, "contract_end": "Various", "contract_value": 1100},
         {"name": "PR agency",        "pct": 15, "contract_end": "2026-06", "contract_value": 825},
         {"name": "Content / Design", "pct": 15, "contract_end": "Various", "contract_value": 825},
     ]},
    {"name": "Facilities & Office",
     "spend": [500,900,1500,2800,4800],
     "budget_2026e": 4500, "concentration": 55, "risk": "High",
     "single_source": False, "lead_time_days": 90, "contract_end": "2028-06",
     "capex_opex": "Opex", "region": "DACH", "supplier_count": 4,
     "po_coverage_pct": 85, "contract_coverage_pct": 95,
     "savings_achieved": 0, "cost_avoidance": 0,
     "contract_value": 5000,
     "suppliers": [
         {"name": "Berlin HQ landlord",  "pct": 55, "contract_end": "2028-06", "contract_value": 2750},
         {"name": "WeWork / Flex space", "pct": 20, "contract_end": "2026-12", "contract_value": 1000},
         {"name": "Cleaning & security", "pct": 15, "contract_end": "2026-09", "contract_value": 750},
         {"name": "Office supplies",     "pct": 10, "contract_end": "Various",  "contract_value": 500},
     ]},
    {"name": "Hardware & Equipment",
     "spend": [300,500,900,1500,2400],
     "budget_2026e": 2200, "concentration": 45, "risk": "Medium",
     "single_source": False, "lead_time_days": 56, "contract_end": "N/A",
     "capex_opex": "Capex", "region": "DACH", "supplier_count": 4,
     "po_coverage_pct": 90, "contract_coverage_pct": 70,
     "savings_achieved": 0, "cost_avoidance": 0,
     "contract_value": 2400,
     "suppliers": [
         {"name": "Apple (MacBooks)",    "pct": 45, "contract_end": "N/A", "contract_value": 1080},
         {"name": "Lenovo",              "pct": 25, "contract_end": "N/A", "contract_value": 600},
         {"name": "Peripherals / Audio", "pct": 20, "contract_end": "N/A", "contract_value": 480},
         {"name": "Monitors / Docking",  "pct": 10, "contract_end": "N/A", "contract_value": 240},
     ]},
    {"name": "Travel & Expenses",
     "spend": [200,500,1100,2000,3200],
     "budget_2026e": 3000, "concentration": 30, "risk": "Low",
     "single_source": False, "lead_time_days": 3, "contract_end": "2026-09",
     "capex_opex": "Opex", "region": "DACH", "supplier_count": 6,
     "po_coverage_pct": 30, "contract_coverage_pct": 60,
     "savings_achieved": 0, "cost_avoidance": 0,
     "contract_value": 3200,
     "suppliers": [
         {"name": "Navan (TMC)",         "pct": 30, "contract_end": "2026-09", "contract_value": 960},
         {"name": "Lufthansa / Airlines","pct": 25, "contract_end": "Rolling",  "contract_value": 800},
         {"name": "Hotels",              "pct": 20, "contract_end": "Rolling",  "contract_value": 640},
         {"name": "Corporate cards",     "pct": 15, "contract_end": "2027-01",  "contract_value": 480},
         {"name": "Rail / local transit","pct": 10, "contract_end": "Rolling",  "contract_value": 320},
     ]},
]

# ── Build DataFrames ──
spend_rows = []
for cat in categories_raw:
    for i, year in enumerate(YEARS):
        spend_rows.append({
            "category":   cat["name"],
            "year":       year,
            "spend":      cat["spend"][i],
            "capex_opex": cat["capex_opex"],
            "region":     cat["region"],
        })
df_spend = pd.DataFrame(spend_rows)

meta_rows = [{
    "category":             c["name"],
    "spend_2026e":          c["spend"][4],
    "spend_2025":           c["spend"][3],
    "budget_2026e":         c["budget_2026e"],
    "concentration":        c["concentration"],
    "risk":                 c["risk"],
    "single_source":        c["single_source"],
    "lead_time_days":       c["lead_time_days"],
    "contract_end":         c["contract_end"],
    "capex_opex":           c["capex_opex"],
    "region":               c["region"],
    "supplier_count":       c["supplier_count"],
    "po_coverage_pct":      c["po_coverage_pct"],
    "contract_coverage_pct":c["contract_coverage_pct"],
    "savings_achieved":     c["savings_achieved"],
    "cost_avoidance":       c["cost_avoidance"],
    "contract_value":       c["contract_value"],
    "cagr":                 round(((c["spend"][4]/c["spend"][0])**(1/4)-1)*100, 1),
    "suppliers":            c["suppliers"],
} for c in categories_raw]
df_meta = pd.DataFrame(meta_rows)

# Derived fields
df_meta["budget_variance"]     = df_meta["spend_2026e"] - df_meta["budget_2026e"]
df_meta["yoy_growth_pct"]      = ((df_meta["spend_2026e"] - df_meta["spend_2025"]) / df_meta["spend_2025"] * 100).round(1)
df_meta["contract_utilisation"] = (df_meta["spend_2026e"] / df_meta["contract_value"] * 100).round(1)

# Headcount & totals
df_hc     = pd.DataFrame({"year": YEARS, "headcount": HEADCOUNT})
df_totals = df_spend.groupby("year")["spend"].sum().reset_index()
df_totals.columns = ["year", "total_spend"]
df_totals = df_totals.merge(df_hc, on="year")
df_totals["yoy_growth"] = df_totals["total_spend"].pct_change() * 100

# Expiring contracts table (top 10 by urgency)
contract_rows = []
today = date.today()
for c in categories_raw:
    for s in c["suppliers"]:
        ce = s["contract_end"]
        if ce in ["N/A", "Various", "Rolling", "Global"]:
            continue
        try:
            exp = datetime.strptime(ce, "%Y-%m").date().replace(day=28)
            days_left = (exp - today).days
            est_spend = round(c["spend"][4] * s["pct"] / 100)
            contract_rows.append({
                "Supplier":         s["name"],
                "Category":         c["name"],
                "Contract Value €K": s["contract_value"],
                "Actual Spend €K":  est_spend,
                "Coverage %":       round(est_spend / s["contract_value"] * 100) if s["contract_value"] > 0 else 0,
                "Expiry":           ce,
                "Days Left":        days_left,
            })
        except:
            pass

df_contracts = pd.DataFrame(contract_rows).sort_values("Days Left").head(10)

# EBITDA impact data
ebitda_rows = [
    {"Initiative": "AWS renegotiation",            "Category": "Cloud & Compute",    "Type": "Savings",        "Impact €K": 800},
    {"Initiative": "Prevented OpenAI price increase","Category": "AI/ML APIs & Data", "Type": "Cost Avoidance", "Impact €K": 320},
    {"Initiative": "SaaS tool consolidation",       "Category": "IT Software & SaaS", "Type": "Savings",        "Impact €K": 280},
    {"Initiative": "Twilio contract consolidation", "Category": "Telecom & Voice",    "Type": "Savings",        "Impact €K": 220},
    {"Initiative": "LinkedIn volume discount",      "Category": "Recruitment & HR",   "Type": "Savings",        "Impact €K": 180},
    {"Initiative": "Deloitte scope reduction",      "Category": "Prof. Services",     "Type": "Savings",        "Impact €K": 150},
    {"Initiative": "Prevented AWS price hike",      "Category": "Cloud & Compute",    "Type": "Cost Avoidance", "Impact €K": 400},
]
df_ebitda = pd.DataFrame(ebitda_rows)

print(f"✅ Data loaded — {len(df_meta)} categories")
print(f"   Total 2026E: €{df_meta['spend_2026e'].sum():,.0f}K")
print(f"   Budget:      €{df_meta['budget_2026e'].sum():,.0f}K")
print(f"   EBITDA Impact: €{df_ebitda['Impact €K'].sum():,.0f}K")
print(f"   Expiring contracts tracked: {len(df_contracts)}")

## 4. Chart Functions

In [ ]:
def chart_spend_stacked(df_spend_filtered, year=None):
    """Stacked area — category contribution over time."""
    fig = go.Figure()
    cats = df_spend_filtered.groupby("category")["spend"].sum().sort_values().index
    for i, cat in enumerate(cats):
        d = df_spend_filtered[df_spend_filtered["category"] == cat]
        fig.add_trace(go.Scatter(
            x=d["year"], y=d["spend"], name=cat[:22], mode="lines",
            stackgroup="one", line=dict(width=0.5, color=COLORS[i % len(COLORS)]),
            hovertemplate=f"<b>{cat}</b><br>Year: %{{x}}<br>Spend: €%{{y:,.0f}}K<extra></extra>",
        ))
    title = f"Spend by Category — {year}" if year else "Spend Evolution by Category"
    fig.update_layout(
        title=title, paper_bgcolor=BG, plot_bgcolor=CARD,
        font=dict(family="Arial, sans-serif", color=TEXT, size=12),
        margin=dict(l=60, r=30, t=50, b=40), height=420,
        xaxis=dict(gridcolor=GRID, linecolor=GRID),
        yaxis=dict(title="Spend (€K)", gridcolor=GRID, linecolor=GRID),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=10), orientation="v", x=1.01, y=1),
    )
    return fig


def chart_category_bars(df_meta_filtered):
    """Horizontal bars: spend per category vs budget."""
    df = df_meta_filtered.sort_values("spend_2026e", ascending=True)
    colors = [RED if row.spend_2026e > row.budget_2026e else NAVY
              for _, row in df.iterrows()]
    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=df["category"], x=df["spend_2026e"], orientation="h",
        name="Actual", marker_color=colors, opacity=0.85,
        text=[f"€{v:,.0f}K" for v in df["spend_2026e"]],
        textposition="outside", textfont=dict(size=9),
    ))
    fig.add_trace(go.Scatter(
        y=df["category"], x=df["budget_2026e"], mode="markers",
        name="Budget", marker=dict(symbol="line-ns", size=12,
                                    color=DIM, line=dict(width=2, color=DIM)),
    ))
    fig.update_layout(
        title="Actual Spend vs Budget (red = over budget)",
        paper_bgcolor=BG, plot_bgcolor=CARD,
        font=dict(family="Arial, sans-serif", color=TEXT, size=12),
        height=420, margin=dict(l=160, r=80, t=50, b=40),
        xaxis=dict(title="Spend (€K)", gridcolor=GRID, linecolor=GRID),
        yaxis=dict(gridcolor=GRID, linecolor=GRID),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=10),
                    orientation="h", y=-0.15),
        barmode="overlay",
    )
    return fig


def chart_risk_bubble(df_meta_filtered):
    """Bottleneck bubble map. Click bubble for drill-down."""
    fig = go.Figure()
    for _, row in df_meta_filtered.iterrows():
        fig.add_trace(go.Scatter(
            x=[row["concentration"]], y=[row["spend_2026e"]],
            mode="markers+text",
            marker=dict(
                size=max(row["lead_time_days"] + 18, 20),
                color=RISK_COLORS.get(row["risk"], DIM),
                opacity=0.75, line=dict(width=1.5, color="white"),
            ),
            text=[row["category"][:16]], textposition="top center",
            textfont=dict(size=9, color=TEXT), name=row["category"],
            customdata=[[row["category"], row["risk"],
                         row["supplier_count"], row["contract_end"]]],
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Concentration: %{x}%<br>Spend: €%{y:,.0f}K<br>"
                f"Lead time: {row['lead_time_days']}d | "
                f"Suppliers: {row['supplier_count']}<br>"
                "Risk: %{customdata[1]}<extra></extra>"
            ),
        ))
    fig.add_shape(type="rect", x0=50, x1=100, y0=0, y1=30000,
                  fillcolor="rgba(192,57,43,0.05)",
                  line=dict(color="rgba(192,57,43,0.4)", dash="dot", width=1.5))
    fig.add_annotation(x=75, y=27000, text="⚠ Danger Zone",
                       showarrow=False, font=dict(size=11, color=RED))
    fig.update_layout(
        title="Bottleneck Map — Click a bubble to see suppliers",
        paper_bgcolor=BG, plot_bgcolor=CARD,
        font=dict(family="Arial, sans-serif", color=TEXT, size=12),
        margin=dict(l=60, r=30, t=50, b=40),
        height=500, showlegend=False,
        xaxis=dict(title="Supplier Concentration (%)", range=[0,105],
                   gridcolor=GRID, linecolor=GRID),
        yaxis=dict(title="Spend (€K)", gridcolor=GRID, linecolor=GRID),
    )
    return fig


def chart_budget_variance(df_meta_filtered):
    """Budget variance per category — red = over, green = under."""
    df = df_meta_filtered.sort_values("budget_variance", ascending=False)
    colors = [RED if v > 0 else GREEN for v in df["budget_variance"]]
    fig = go.Figure(go.Bar(
        y=df["category"], x=df["budget_variance"], orientation="h",
        marker_color=colors,
        text=[f"€{v:+,.0f}K" for v in df["budget_variance"]],
        textposition="outside",
    ))
    fig.add_vline(x=0, line_width=1.5, line_color=TEXT)
    fig.update_layout(
        title="Budget Variance (Actual − Budget)",
        paper_bgcolor=BG, plot_bgcolor=CARD,
        font=dict(family="Arial, sans-serif", color=TEXT, size=12),
        height=400, margin=dict(l=160, r=80, t=50, b=40),
        xaxis=dict(title="Variance €K", gridcolor=GRID, linecolor=GRID),
        yaxis=dict(gridcolor=GRID, linecolor=GRID),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=10)),
    )
    return fig


def chart_ebitda_waterfall(df_ebitda):
    """Waterfall chart showing EBITDA contribution by initiative."""
    df = df_ebitda.copy()
    total = df["Impact €K"].sum()
    fig = go.Figure(go.Waterfall(
        orientation="v",
        measure=["relative"] * len(df) + ["total"],
        x=list(df["Initiative"]) + ["Total Impact"],
        y=list(df["Impact €K"]) + [total],
        text=[f"€{v:,.0f}K" for v in list(df["Impact €K"]) + [total]],
        textposition="outside",
        connector=dict(line=dict(color=GRID, width=1)),
        increasing=dict(marker=dict(color=GREEN)),
        decreasing=dict(marker=dict(color=RED)),
        totals=dict(marker=dict(color=NAVY)),
    ))
    fig.update_layout(
        title=f"Procurement EBITDA Impact — Total €{total:,.0f}K",
        paper_bgcolor=BG, plot_bgcolor=CARD,
        font=dict(family="Arial, sans-serif", color=TEXT, size=11),
        height=420, margin=dict(l=60, r=30, t=50, b=100),
        xaxis=dict(gridcolor=GRID, linecolor=GRID, tickangle=-35),
        yaxis=dict(title="€K", gridcolor=GRID, linecolor=GRID),
        legend=dict(bgcolor="rgba(0,0,0,0)"),
    )
    return fig


def chart_cagr(df_meta_filtered):
    """CAGR per category."""
    df = df_meta_filtered.sort_values("cagr", ascending=True)
    colors = [RED if v > 40 else YELLOW if v > 25 else NAVY for v in df["cagr"]]
    fig = go.Figure(go.Bar(
        y=df["category"], x=df["cagr"], orientation="h",
        marker_color=colors,
        text=[f"{v:.1f}%" for v in df["cagr"]], textposition="outside",
    ))
    fig.update_layout(
        title="Category CAGR (2022→2026E)",
        paper_bgcolor=BG, plot_bgcolor=CARD,
        font=dict(family="Arial, sans-serif", color=TEXT, size=12),
        height=400, margin=dict(l=160, r=60, t=50, b=40),
        xaxis=dict(title="CAGR %", gridcolor=GRID, linecolor=GRID),
        yaxis=dict(gridcolor=GRID, linecolor=GRID),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=10)),
    )
    return fig


def chart_treemap(df_meta_filtered):
    """Treemap: size = spend, color = concentration."""
    df = df_meta_filtered.copy()
    df["label"] = df.apply(
        lambda r: f"{r['category']}<br>€{r['spend_2026e']:,.0f}K", axis=1)
    fig = px.treemap(df, path=["label"], values="spend_2026e", color="concentration",
                     color_continuous_scale=["#1A7A4A","#B8860B","#C0392B"],
                     range_color=[0, 80])
    fig.update_layout(
        title="Spend × Concentration", paper_bgcolor=BG,
        font=dict(color=TEXT, size=12), height=420,
        coloraxis_colorbar=dict(title="Conc %"),
    )
    return fig


def chart_capex_opex(df_spend_filtered):
    """Capex vs Opex grouped bar."""
    df = df_spend_filtered.groupby(["year","capex_opex"])["spend"].sum().reset_index()
    fig = go.Figure()
    for ct, col in [("Opex", NAVY), ("Capex", NAVY2)]:
        d = df[df["capex_opex"] == ct]
        fig.add_trace(go.Bar(
            x=d["year"], y=d["spend"], name=ct, marker_color=col, opacity=0.85,
            text=[f"€{v:,.0f}K" for v in d["spend"]], textposition="outside",
        ))
    fig.update_layout(
        title="Capex vs Opex",
        paper_bgcolor=BG, plot_bgcolor=CARD,
        font=dict(family="Arial, sans-serif", color=TEXT, size=12),
        margin=dict(l=60, r=30, t=50, b=40),
        height=350, barmode="group",
        xaxis=dict(gridcolor=GRID, linecolor=GRID),
        yaxis=dict(title="Spend (€K)", gridcolor=GRID, linecolor=GRID),
        legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=10)),
    )
    return fig

print("✅ Chart functions defined")

## 5. Battery / Vertical Progress Diagrams
Vertical battery-style KPI health indicators.

In [ ]:
def vertical_battery(label, value, target, unit="%", lower_is_better=False):
    """
    Vertical battery diagram as HTML.
    Shows current value vs target with color-coded fill.
    lower_is_better=True for maverick spend (lower = healthier).
    """
    color = traffic_color(value, target, lower_is_better)

    if lower_is_better:
        # For maverick: fill from top (bad = full, good = empty)
        fill_pct = min(value / target * 100, 100) if target > 0 else 0
        fill_pct = min(fill_pct * 2, 100)  # scale so target = 50% fill
    else:
        fill_pct = min(value / target * 100, 100) if target > 0 else 0

    empty_pct = 100 - fill_pct

    return pn.pane.HTML(f"""
    <div style="display:flex; flex-direction:column; align-items:center;
                padding:12px 16px; background:{CARD}; border-radius:10px;
                border:1px solid {GRID}; min-width:110px; text-align:center;">

        <div style="font-size:11px; color:{DIM}; text-transform:uppercase;
                    letter-spacing:0.06em; margin-bottom:10px; font-weight:500;">
            {label}
        </div>

        <!-- Battery body -->
        <div style="position:relative; width:44px; height:160px;
                    border:2px solid {color}; border-radius:4px;
                    background:{BG}; overflow:hidden;">

            <!-- Battery cap -->
            <div style="position:absolute; top:-8px; left:50%;
                        transform:translateX(-50%);
                        width:16px; height:8px;
                        background:{color}; border-radius:2px 2px 0 0;">
            </div>

            <!-- Empty portion (top) -->
            <div style="height:{empty_pct}%; background:rgba(0,0,0,0.04);">
            </div>

            <!-- Filled portion (bottom) -->
            <div style="height:{fill_pct}%; background:{color}; opacity:0.85;">
            </div>

            <!-- Value label inside battery -->
            <div style="position:absolute; top:50%; left:50%;
                        transform:translate(-50%,-50%);
                        font-size:13px; font-weight:700;
                        color:{'white' if fill_pct > 40 else TEXT};">
                {value}{unit}
            </div>
        </div>

        <!-- Target label -->
        <div style="margin-top:8px; font-size:11px; color:{DIM};">
            Target: {target}{unit}
        </div>

        <!-- Status label -->
        <div style="margin-top:4px; font-size:11px; font-weight:600; color:{color};">
            {"✓ On track" if color == GREEN else "⚠ Watch" if color == YELLOW else "✗ Off track"}
        </div>
    </div>
    """)

print("✅ Battery diagram ready")

## 6. KPI Cards, Drill-down Panels & Contract Table

In [ ]:
def kpi_card(title, value, subtitle="", color=NAVY, clickable=False):
    """Styled KPI card. clickable=True adds a pointer cursor hint."""
    cursor = "pointer" if clickable else "default"
    border = f"2px solid {NAVY}" if clickable else f"1px solid {GRID}"
    return pn.pane.HTML(f"""
    <div style="background:{CARD}; border-radius:8px; padding:16px 20px;
                border:{border}; min-width:140px; text-align:center;
                cursor:{cursor};">
        <div style="color:{DIM}; font-size:11px; text-transform:uppercase;
                    letter-spacing:0.08em; margin-bottom:6px;">{title}</div>
        <div style="color:{color}; font-size:28px; font-weight:600;
                    margin:4px 0; line-height:1.1;">{value}</div>
        <div style="color:{DIM}; font-size:11px; margin-top:4px;">{subtitle}</div>
        {"<div style=\"font-size:10px; color:" + NAVY2 + "; margin-top:6px;\">▼ click for detail</div>" if clickable else ""}
    </div>
    """, sizing_mode="stretch_width")


def render_drill_down(category_name):
    """Supplier detail panel for clicked bubble."""
    row = df_meta[df_meta["category"] == category_name]
    if row.empty:
        return pn.pane.Markdown("No data.")
    row = row.iloc[0]
    suppliers = row["suppliers"]
    total_spend = row["spend_2026e"]
    risk_color = RISK_COLORS.get(row["risk"], DIM)

    rows_html = ""
    for i, s in enumerate(suppliers):
        est_spend = round(total_spend * s["pct"] / 100)
        bg = CARD if i % 2 == 0 else BG
        rows_html += f"""
        <tr style="background:{bg};">
            <td style="padding:8px 12px; font-size:13px;">{s["name"]}</td>
            <td style="padding:8px 12px; font-size:13px; text-align:right;">{s["pct"]}%</td>
            <td style="padding:8px 12px; font-size:13px; text-align:right;
                       color:{NAVY}; font-weight:500;">€{est_spend:,.0f}K</td>
            <td style="padding:8px 12px; font-size:12px; color:{DIM};">{s["contract_end"]}</td>
        </tr>"""

    html = f"""
    <div style="background:{BG}; border:1px solid {GRID}; border-radius:8px;
                padding:16px 20px; margin-top:12px;">
        <div style="display:flex; align-items:center; gap:10px; margin-bottom:10px;">
            <span style="font-size:15px; font-weight:600; color:{NAVY};">
                {category_name}</span>
            <span style="font-size:12px; color:{risk_color}; font-weight:500;">
                ● {row["risk"]} risk</span>
            <span style="font-size:12px; color:{DIM};">
                | {row["supplier_count"]} suppliers | Conc. {row["concentration"]}%
                | Lead time {row["lead_time_days"]}d</span>
        </div>
        <table style="width:100%; border-collapse:collapse;
                      border:0.5px solid {GRID}; border-radius:6px;">
            <thead>
                <tr style="background:{NAVY}; color:white;">
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Supplier</th>
                    <th style="padding:8px 12px; text-align:right; font-size:12px;">Share</th>
                    <th style="padding:8px 12px; text-align:right; font-size:12px;">Est. Spend</th>
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Contract End</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
        <div style="margin-top:6px; font-size:11px; color:{DIM};">
            * Spend estimated from % split of 2026E total (€{total_spend:,.0f}K)
        </div>
    </div>"""
    return pn.pane.HTML(html, sizing_mode="stretch_width")


def render_contracts_table(df_contracts):
    """Expiring contracts table with traffic light coloring."""
    rows_html = ""
    for i, row in df_contracts.iterrows():
        days = row["Days Left"]
        if days <= 60:
            dot_color = RED
            urgency = "⚠ Urgent"
        elif days <= 180:
            dot_color = YELLOW
            urgency = "Soon"
        else:
            dot_color = GREEN
            urgency = "OK"

        cov_color = RED if row["Coverage %"] > 100 else GREEN if row["Coverage %"] > 70 else YELLOW
        bg = CARD if int(i) % 2 == 0 else BG

        rows_html += f"""
        <tr style="background:{bg};">
            <td style="padding:8px 12px; font-size:13px; font-weight:500;">{row["Supplier"]}</td>
            <td style="padding:8px 12px; font-size:12px; color:{DIM};">{row["Category"]}</td>
            <td style="padding:8px 12px; font-size:13px; text-align:right;">€{row["Contract Value €K"]:,.0f}K</td>
            <td style="padding:8px 12px; font-size:13px; text-align:right;">€{row["Actual Spend €K"]:,.0f}K</td>
            <td style="padding:8px 12px; font-size:13px; text-align:center; color:{cov_color}; font-weight:600;">{row["Coverage %"]}%</td>
            <td style="padding:8px 12px; font-size:12px;">{row["Expiry"]}</td>
            <td style="padding:8px 12px; font-size:12px; color:{dot_color}; font-weight:600;">{urgency} ({days}d)</td>
        </tr>"""

    return pn.pane.HTML(f"""
    <div style="background:{BG}; border:1px solid {GRID}; border-radius:8px;
                padding:16px 20px; margin-top:8px;">
        <div style="font-size:15px; font-weight:600; color:{NAVY}; margin-bottom:10px;">
            Top 10 Expiring Contracts — Contract Value vs Actual Spend
        </div>
        <table style="width:100%; border-collapse:collapse;">
            <thead>
                <tr style="background:{NAVY}; color:white;">
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Supplier</th>
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Category</th>
                    <th style="padding:8px 12px; text-align:right; font-size:12px;">Contract €K</th>
                    <th style="padding:8px 12px; text-align:right; font-size:12px;">Actual €K</th>
                    <th style="padding:8px 12px; text-align:center; font-size:12px;">Coverage</th>
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Expiry</th>
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Status</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
    </div>
    """, sizing_mode="stretch_width")


def render_ebitda_panel():
    """EBITDA impact detail panel."""
    total_savings   = df_ebitda[df_ebitda["Type"]=="Savings"]["Impact €K"].sum()
    total_avoidance = df_ebitda[df_ebitda["Type"]=="Cost Avoidance"]["Impact €K"].sum()
    total           = df_ebitda["Impact €K"].sum()

    rows_html = ""
    for i, row in df_ebitda.iterrows():
        col = GREEN if row["Type"] == "Savings" else NAVY2
        bg  = CARD if int(i) % 2 == 0 else BG
        rows_html += f"""
        <tr style="background:{bg};">
            <td style="padding:8px 12px; font-size:13px;">{row["Initiative"]}</td>
            <td style="padding:8px 12px; font-size:12px; color:{DIM};">{row["Category"]}</td>
            <td style="padding:8px 12px; font-size:12px; color:{col}; font-weight:500;">{row["Type"]}</td>
            <td style="padding:8px 12px; font-size:13px; text-align:right;
                       color:{GREEN}; font-weight:600;">+€{row["Impact €K"]:,.0f}K</td>
        </tr>"""

    return pn.pane.HTML(f"""
    <div style="background:{BG}; border:1px solid {GRID}; border-radius:8px;
                padding:16px 20px; margin-top:8px;">
        <div style="display:flex; gap:24px; margin-bottom:14px; flex-wrap:wrap;">
            <div style="text-align:center;">
                <div style="font-size:11px; color:{DIM}; text-transform:uppercase;">Negotiated Savings</div>
                <div style="font-size:24px; font-weight:700; color:{GREEN};">€{total_savings:,.0f}K</div>
            </div>
            <div style="text-align:center;">
                <div style="font-size:11px; color:{DIM}; text-transform:uppercase;">Cost Avoidance</div>
                <div style="font-size:24px; font-weight:700; color:{NAVY2};">€{total_avoidance:,.0f}K</div>
            </div>
            <div style="text-align:center;">
                <div style="font-size:11px; color:{DIM}; text-transform:uppercase;">Total EBITDA Impact</div>
                <div style="font-size:24px; font-weight:700; color:{NAVY};">€{total:,.0f}K</div>
            </div>
        </div>
        <table style="width:100%; border-collapse:collapse;">
            <thead>
                <tr style="background:{NAVY}; color:white;">
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Initiative</th>
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Category</th>
                    <th style="padding:8px 12px; text-align:left; font-size:12px;">Type</th>
                    <th style="padding:8px 12px; text-align:right; font-size:12px;">Impact</th>
                </tr>
            </thead>
            <tbody>{rows_html}</tbody>
        </table>
    </div>
    """, sizing_mode="stretch_width")

print("✅ Cards, panels, tables ready")

## 7. Launch Dashboard 🚀

**Structure:**
- Header: 5 KPI cards (click EBITDA, Contract Coverage, Maverick for detail panels)
- Section 1: Spend Overview — stacked area + actual vs budget bars
- Section 2: Risk — bubble map + expiring contracts table
- Section 3: Procurement Health — 4 vertical batteries + PO vs Invoice
- Section 4: EBITDA Impact — waterfall chart + budget variance
- Section 5: Data Explorer
- Deep Dive tab: CAGR, treemap, capex/opex


In [ ]:
# ── Sidebar widgets ──
year_select = pn.widgets.Select(
    name="Year", options=["All years"] + YEARS, value="All years", width=250)
file_input = pn.widgets.FileInput(accept=".csv,.xlsx,.xls", name="Upload spend data")
dataset_label = pn.pane.Markdown("**Dataset:** Default", styles={"color": TEXT})
api_key_input = pn.widgets.PasswordInput(
    name="Claude API key (optional)", placeholder="sk-ant-...", width=250)
export_btn = pn.widgets.Button(
    name="📥 Export CFO Report", button_type="primary", width=250)
status_log = pn.pane.Markdown("", styles={"color": DIM, "font-size": "11px"})

# ── Content containers ──
kpi_row       = pn.Row(sizing_mode="stretch_width")
detail_panel  = pn.Column(sizing_mode="stretch_width")  # KPI card drill-down
chart_s1      = pn.Column(sizing_mode="stretch_width")  # Section 1
chart_s2      = pn.Column(sizing_mode="stretch_width")  # Section 2
chart_s3      = pn.Column(sizing_mode="stretch_width")  # Section 3
chart_s4      = pn.Column(sizing_mode="stretch_width")  # Section 4
drill_panel   = pn.Column(sizing_mode="stretch_width")  # Bubble drill-down
chart_deep    = pn.Column(sizing_mode="stretch_width")  # Deep dive tab

data_preview = pn.widgets.Tabulator(
    df_meta.drop(columns=["suppliers"]),
    sizing_mode="stretch_width", height=300,
    theme="bootstrap", page_size=12,
)


def get_filtered(year_val):
    if year_val == "All years":
        return df_meta.copy(), df_spend.copy()
    ds = df_spend[df_spend["year"] == year_val].copy()
    dm = df_meta.copy()
    ys = ds.groupby("category")["spend"].sum()
    dm["spend_2026e"] = dm["category"].map(ys).fillna(0)
    dm["budget_variance"] = dm["spend_2026e"] - dm["budget_2026e"]
    return dm, ds


def update_dashboard(year_val=None):
    if year_val is None:
        year_val = year_select.value
    dm, ds = get_filtered(year_val)
    year_label = str(year_val) if year_val != "All years" else "2026E"

    total     = dm["spend_2026e"].sum()
    prev      = dm["spend_2025"].sum()
    yoy       = round((total - prev) / prev * 100, 1) if prev > 0 else 0
    ebitda    = df_ebitda["Impact €K"].sum()
    maverick  = 12
    contract_cov = 80

    # ── KPI cards ──
    kpi_row.clear()
    kpi_row.extend([
        kpi_card("Total Spend",        f"€{total/1000:.1f}M",  year_label),
        kpi_card("YoY Growth",         f"+{yoy:.0f}%",         "vs prior year",
                 RED if yoy > 30 else YELLOW if yoy > 15 else GREEN),
        kpi_card("EBITDA Impact",      f"€{ebitda:,.0f}K",     "savings + avoidance",
                 GREEN, clickable=True),
        kpi_card("Contract Coverage",  f"{contract_cov}%",     "target 100%",
                 traffic_color(contract_cov, 100), clickable=True),
        kpi_card("Maverick Spend",     f"{maverick}%",         "target 5%",
                 traffic_color(maverick, 5, lower_is_better=True), clickable=True),
    ])

    # ── Section 1: Spend Overview ──
    chart_s1.clear()
    chart_s1.extend([
        pn.pane.HTML(f"<h3 style='color:{NAVY}; margin:16px 0 4px 0;'>📊 Spend Overview</h3>",
                     sizing_mode="stretch_width"),
        pn.Row(
            pn.pane.Plotly(chart_spend_stacked(ds, year_val if year_val != "All years" else None),
                           sizing_mode="stretch_width"),
            pn.pane.Plotly(chart_category_bars(dm), sizing_mode="stretch_width"),
        ),
    ])

    # ── Section 2: Risk ──
    chart_s2.clear()
    chart_s2.extend([
        pn.pane.HTML(f"<h3 style='color:{NAVY}; margin:16px 0 4px 0;'>⚠ Risk & Bottlenecks</h3>",
                     sizing_mode="stretch_width"),
        pn.pane.Plotly(chart_risk_bubble(dm), sizing_mode="stretch_width"),
        drill_panel,
        render_contracts_table(df_contracts),
    ])

    # ── Section 3: Procurement Health ──
    po_avg     = round(dm["po_coverage_pct"].mean())
    cc_avg     = round(dm["contract_coverage_pct"].mean())
    spm_pct    = 60

    chart_s3.clear()
    chart_s3.extend([
        pn.pane.HTML(f"<h3 style='color:{NAVY}; margin:16px 0 4px 0;'>🏥 Procurement Health</h3>",
                     sizing_mode="stretch_width"),
        pn.Row(
            vertical_battery("PO Coverage",          po_avg,   100),
            vertical_battery("Contract Coverage",     cc_avg,   100),
            vertical_battery("Maverick Spend",        maverick,   5, lower_is_better=True),
            vertical_battery("Spend Under Mgmt",      spm_pct,   80),
            sizing_mode="stretch_width",
        ),
    ])

    # ── Section 4: EBITDA Impact ──
    chart_s4.clear()
    chart_s4.extend([
        pn.pane.HTML(f"<h3 style='color:{NAVY}; margin:16px 0 4px 0;'>💶 EBITDA Impact</h3>",
                     sizing_mode="stretch_width"),
        pn.Row(
            pn.pane.Plotly(chart_ebitda_waterfall(df_ebitda), sizing_mode="stretch_width"),
            pn.pane.Plotly(chart_budget_variance(dm),          sizing_mode="stretch_width"),
        ),
        render_ebitda_panel(),
    ])

    # ── Deep Dive ──
    chart_deep.clear()
    chart_deep.extend([
        pn.pane.HTML(f"<h3 style='color:{NAVY}; margin:16px 0 4px 0;'>🔬 Deep Dive</h3>",
                     sizing_mode="stretch_width"),
        pn.Row(
            pn.pane.Plotly(chart_cagr(dm),         sizing_mode="stretch_width"),
            pn.pane.Plotly(chart_capex_opex(ds),    sizing_mode="stretch_width"),
        ),
        pn.pane.Plotly(chart_treemap(dm), sizing_mode="stretch_width"),
    ])

    data_preview.value = dm.drop(columns=["suppliers"])


# ── Year selector ──
def on_year_change(event):
    update_dashboard(event.new)
year_select.param.watch(on_year_change, "value")


# ── KPI card click detail panels ──
ebitda_btn   = pn.widgets.Button(name="💶 EBITDA Detail",        button_type="light", width=160)
contract_btn = pn.widgets.Button(name="📋 Contract Detail",      button_type="light", width=160)
maverick_btn = pn.widgets.Button(name="⚠ Maverick Detail",       button_type="light", width=160)

def show_ebitda(event):
    detail_panel.clear()
    detail_panel.append(render_ebitda_panel())

def show_contracts(event):
    detail_panel.clear()
    detail_panel.append(render_contracts_table(df_contracts))

def show_maverick(event):
    detail_panel.clear()
    detail_panel.append(pn.pane.HTML(f"""
    <div style="background:{CARD}; border:1px solid {GRID}; border-radius:8px; padding:16px 20px;">
        <div style="font-size:15px; font-weight:600; color:{NAVY}; margin-bottom:8px;">
            Maverick Spend Analysis</div>
        <p style="color:{TEXT}; font-size:13px;">
            Current maverick spend is estimated at <b style="color:{RED};">12%</b>
            of total spend — target is <b>5%</b>.<br><br>
            Main sources of maverick spend:<br>
            • Travel & Expenses — bookings outside Navan (30% non-compliant)<br>
            • Marketing — ad hoc agency spend without PO (60% non-PO)<br>
            • IT Software — shadow IT subscriptions (45% non-contracted)<br>
            • Recruitment — direct agency bookings bypassing procurement<br><br>
            <b>Action:</b> Enforce PO requirement for categories above €5K.
            Estimated recovery: €{round(df_meta["spend_2026e"].sum()*0.07/1000,1)}M → €{round(df_meta["spend_2026e"].sum()*0.05/1000,1)}M.
        </p>
    </div>
    """, sizing_mode="stretch_width"))

ebitda_btn.on_click(show_ebitda)
contract_btn.on_click(show_contracts)
maverick_btn.on_click(show_maverick)


# ── Bubble click ──
def on_bubble_click(event):
    if event.new and "points" in event.new and event.new["points"]:
        pt = event.new["points"][0]
        cat_name = pt["customdata"][0] if "customdata" in pt and pt["customdata"] else                    pt.get("text","").strip()
        drill_panel.clear()
        drill_panel.append(render_drill_down(cat_name))

bubble_pane = pn.pane.Plotly(chart_risk_bubble(df_meta), sizing_mode="stretch_width")
bubble_pane.param.watch(on_bubble_click, "selected_data")


# ── File upload ──
def handle_upload(event):
    if file_input.value is None: return
    try:
        raw = io.BytesIO(file_input.value)
        fname = file_input.filename
        df_new = pd.read_csv(raw) if fname.endswith(".csv") else pd.read_excel(raw)
        status_log.object = f"✅ **{fname}** — {len(df_new)} rows"
        dataset_label.object = f"**Dataset:** {fname}"
        data_preview.value = df_new.head(50)
    except Exception as e:
        status_log.object = f"❌ {str(e)}"

file_input.param.watch(handle_upload, "value")


# ── Export ──
def handle_export(event):
    try:
        buf = io.BytesIO()
        with pd.ExcelWriter(buf, engine="openpyxl") as w:
            df_meta.drop(columns=["suppliers"]).to_excel(w, sheet_name="Category Summary", index=False)
            df_spend.to_excel(w, sheet_name="Spend Over Time", index=False)
            df_contracts.to_excel(w, sheet_name="Expiring Contracts", index=False)
            df_ebitda.to_excel(w, sheet_name="EBITDA Impact", index=False)
        ts = datetime.now().strftime("%Y%m%d_%H%M")
        dl = pn.widgets.FileDownload(
            file=io.BytesIO(buf.getvalue()),
            filename=f"CFO_Report_{ts}.xlsx",
            button_type="success", label="⬇ Download")
        status_log.object = f"✅ Report ready"
        sidebar_col.append(dl)
    except Exception as e:
        status_log.object = f"❌ {str(e)}"

export_btn.on_click(handle_export)

# ── Initial render ──
update_dashboard()

# ── Header ──
header = pn.pane.HTML(f"""
<div style="background:{NAVY}; padding:20px 28px; border-radius:8px; margin-bottom:16px;">
    <h1 style="color:white; margin:0; font-size:24px; font-weight:600;">SpendLens</h1>
    <p style="color:rgba(255,255,255,0.7); margin:4px 0 0 0; font-size:13px;">
        Procurement Intelligence — Upload any spend data for instant insights
    </p>
</div>
""", sizing_mode="stretch_width")

# ── Detail panel buttons ──
detail_buttons = pn.Row(
    pn.pane.HTML(f"<span style='color:{DIM}; font-size:12px;'>Show detail:</span>"),
    ebitda_btn, contract_btn, maverick_btn,
    sizing_mode="stretch_width",
)

# ── Sidebar ──
sidebar_col = pn.Column(
    pn.pane.HTML(f"<div style='color:{NAVY}; font-weight:600; font-size:13px;'>YEAR</div>"),
    year_select, pn.layout.Divider(),
    pn.pane.HTML(f"<div style='color:{NAVY}; font-weight:600; font-size:13px;'>DATA SOURCE</div>"),
    file_input, dataset_label, pn.layout.Divider(),
    pn.pane.HTML(f"<div style='color:{NAVY}; font-weight:600; font-size:13px;'>AI SETTINGS</div>"),
    api_key_input, pn.layout.Divider(),
    pn.pane.HTML(f"<div style='color:{NAVY}; font-weight:600; font-size:13px;'>REPORTS</div>"),
    export_btn, pn.layout.Divider(),
    status_log, width=280,
)

# ── Export after sidebar defined ──
export_btn.on_click(handle_export)

# ── Main tabs ──
main_tab = pn.Column(
    header,
    kpi_row,
    detail_buttons,
    detail_panel,
    pn.layout.Divider(),
    chart_s1, pn.layout.Divider(),
    chart_s2, pn.layout.Divider(),
    chart_s3, pn.layout.Divider(),
    chart_s4, pn.layout.Divider(),
    pn.pane.HTML(f"<h3 style='color:{NAVY}; margin:16px 0 4px 0;'>📋 Data Explorer</h3>"),
    data_preview,
    sizing_mode="stretch_width",
)

tabs = pn.Tabs(
    ("Dashboard", main_tab),
    ("Deep Dive", chart_deep),
    sizing_mode="stretch_width",
)

template = pn.template.FastListTemplate(
    title="SpendLens",
    sidebar=sidebar_col,
    main=[tabs],
    accent_base_color=NAVY,
    header_background=NAVY,
    background_color=BG,
    theme="default",
)

template.show()
print("\n🚀 Dashboard open — http://localhost:5006")

## 8. AI Column Mapper
*(maps any upload to standard schema)*

In [ ]:
STANDARD_SCHEMA = {
    "category":    ["category","spend_category","warengruppe","kategorie","commodity"],
    "supplier":    ["supplier","vendor","provider","lieferant","kreditor","company"],
    "spend":       ["spend","amount","total","betrag","invoice_total","rechnungsbetrag"],
    "date":        ["date","invoice_date","datum","posting_date","buchungsdatum"],
    "contract_end":["contract_end","expiry_date","end_date","vertragslaufzeit"],
    "region":      ["region","country","location","land"],
    "po_number":   ["po_number","purchase_order","bestellnummer"],
    "capex_opex":  ["capex_opex","capex","opex","cost_type"],
    "budget":      ["budget","budget_2026e","planned_spend","plan"],
}

def rule_based_mapping(columns):
    mapping = {}
    normalized = {c: c.strip().lower().replace(" ","_").replace("-","_") for c in columns}
    for orig, norm in normalized.items():
        matched = False
        for std, syns in STANDARD_SCHEMA.items():
            if norm in syns or any(s in norm for s in syns):
                mapping[orig] = std; matched = True; break
        if not matched:
            mapping[orig] = None
    return mapping

print("✅ Column mapper ready")

## 9. Data Cleanup Engine

In [ ]:
def clean_column_names(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[^\w\s]","",regex=True)
                  .str.replace(r"\s+","_",regex=True))
    return df

def fix_spend_column(series):
    def clean(val):
        if pd.isna(val): return None
        s = str(val).strip()
        neg = s.startswith("(") and s.endswith(")")
        if neg: s = s[1:-1]
        s = re.sub(r"[€$£¥\s]","",s)
        if re.search(r"\d\.\d{3},\d",s): s = s.replace(".","").replace(",",".")
        else: s = s.replace(",","")
        try: return -float(s) if neg else float(s)
        except: return None
    return series.apply(clean)

def full_cleanup(df):
    df = clean_column_names(df)
    df = df.dropna(how="all").drop_duplicates()
    for col in [c for c in df.columns if any(k in c for k in ["spend","amount","betrag"])]:
        df[col] = fix_spend_column(df[col])
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].fillna("Unknown")
    return df

print("✅ Cleanup engine ready")